## 1. 安装依赖

In [15]:
!pip install datasets transformers torch soundfile librosa jiwer tqdm langdetect

## 2. 导入库

In [16]:
import torch
import numpy as np
import io
import re
import json
import csv
from datetime import datetime
from pathlib import Path
import soundfile as sf
from datasets import load_dataset
from transformers import pipeline
from jiwer import wer, cer
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# 用于语言检测
from langdetect import detect, LangDetectException

print("✓ 库导入完成")

✓ 库导入完成


## 3. 配置参数

In [17]:
# ==================== 项目配置 ====================

# 数据集设置
NUM_SAMPLES = 2000  # 改为 2000 条
DATASET_SPLIT = "test"

# 模型设置
MODEL_NAME = "openai/whisper-small"

# 语言设置（重要！）
FORCE_ENGLISH = False  # 强制英文识别
FILTER_NON_ENGLISH = False  # 过滤非英文结果

# 输出文件
OUTPUT_JSON = "transcription_results_2000_english.json"
OUTPUT_CSV = "transcription_results_2000_english.csv"

# 进度保存
CHECKPOINT_FILE = "checkpoint_2000.json"
SAVE_EVERY = 100

# 断点续传
RESUME = True

print("="*80)
print("项目配置")
print("="*80)
print(f"样本数量: {NUM_SAMPLES}")
print(f"数据集: {DATASET_SPLIT}")
print(f"模型: {MODEL_NAME}")
print(f"强制英文: {'是' if FORCE_ENGLISH else '否'}")
print(f"过滤非英文: {'是' if FILTER_NON_ENGLISH else '否'}")
print(f"输出文件: {OUTPUT_JSON}, {OUTPUT_CSV}")
print("="*80)

项目配置
样本数量: 2000
数据集: test
模型: openai/whisper-small
强制英文: 否
过滤非英文: 否
输出文件: transcription_results_2000_english.json, transcription_results_2000_english.csv


## 4. 文本规范化和语言检测函数

In [18]:
def normalize_text(text):
    """
    规范化文本用于 WER/CER 计算
    """
    if not text:
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def is_english(text, threshold=0.7):
    """
    检测文本是否为英文
    
    Args:
        text: 输入文本
        threshold: 英文字符占比阈值
    
    Returns:
        bool: 是否为英文
    """
    if not text or len(text.strip()) < 3:
        return False
    
    # 方法1: 检查英文字母占比
    english_chars = sum(1 for c in text if c.isalpha() and ord(c) < 128)
    total_chars = sum(1 for c in text if c.isalpha())
    
    if total_chars == 0:
        return False
    
    english_ratio = english_chars / total_chars
    
    # 方法2: 使用 langdetect（更准确但可能失败）
    try:
        detected_lang = detect(text)
        is_en_detected = detected_lang == 'en'
    except LangDetectException:
        is_en_detected = None
    
    # 综合判断
    if is_en_detected is not None:
        # 如果 langdetect 有结果，优先使用
        return is_en_detected
    else:
        # 否则使用字符占比
        return english_ratio >= threshold


# 测试
print("测试语言检测:")
test_cases = [
    ("Hello, how are you?", True),
    ("yng nghymru yng nghymru", False),
    ("This is English text", True),
    ("你好世界", False)
]

for text, expected in test_cases:
    result = is_english(text)
    status = "✓" if result == expected else "✗"
    print(f"  {status} '{text}' → {result}")

测试语言检测:
  ✓ 'Hello, how are you?' → True
  ✓ 'yng nghymru yng nghymru' → False
  ✓ 'This is English text' → True
  ✓ '你好世界' → False


## 5. 检查点管理

In [19]:
def save_checkpoint(results, index):
    checkpoint = {
        'last_index': index,
        'total_processed': len(results),
        'timestamp': datetime.now().isoformat(),
        'results': results
    }
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(checkpoint, f, ensure_ascii=False)

def load_checkpoint():
    try:
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        return None

print("✓ 检查点管理函数已定义")

✓ 检查点管理函数已定义


## 6. 加载数据集

In [20]:
from datasets import Audio

print("正在加载数据集...")
full_dataset = load_dataset("speechbrain/LoquaciousSet", "small", split=DATASET_SPLIT)

# This ensures the 'audio' key exists and is resampled to 16kHz automatically
full_dataset = full_dataset.cast_column("audio", Audio(sampling_rate=16000))

dataset = full_dataset.select(range(min(NUM_SAMPLES, len(full_dataset))))

正在加载数据集...


In [21]:
# Test a single sample
test_sample = dataset[0]
text, eng = transcribe_sample(test_sample)
print(f"Sample ID: {test_sample['ID']}")
print(f"Transcription: {text}")
print(f"Is English: {eng}")

Sample ID: 20180911-0900-PLENARY-en_20180911-20:17:50_2
Transcription: 
Is English: False


In [ ]:
'''print("正在加载数据集...")
full_dataset = load_dataset(
    "speechbrain/LoquaciousSet",
    "small",
    split=DATASET_SPLIT
)

# 选取前 2000 个样本
dataset = full_dataset.select(range(min(NUM_SAMPLES, len(full_dataset))))

print(f"✓ 数据集加载完成")
print(f"  总可用样本: {len(full_dataset)}")
print(f"  选取样本: {len(dataset)}")
print(f"  数据集分割: {DATASET_SPLIT}")'''

正在加载数据集...


✓ 数据集加载完成
  总可用样本: 8087
  选取样本: 2000
  数据集分割: test


## 7. 初始化模型（带英文限制）

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"使用设备: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"\n正在加载模型: {MODEL_NAME}...")

# 重要：设置语言参数
if FORCE_ENGLISH:
    print("⚠️  强制英文模式：将 'language' 设置为 'english'")
    transcriber = pipeline(
        "automatic-speech-recognition",
        model=MODEL_NAME,
        device=0 if device == "cuda" else -1,
        chunk_length_s=30,
        generate_kwargs={"language": "english"}  # 关键：强制英文
    )
else:
    transcriber = pipeline(
        "automatic-speech-recognition",
        model=MODEL_NAME,
        device=0 if device == "cuda" else -1,
        chunk_length_s=30
    )

print("✓ 模型加载完成!")

使用设备: cpu

正在加载模型: openai/whisper-small...


Loading weights: 100%|██████████| 479/479 [00:00<00:00, 770.83it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]   
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


✓ 模型加载完成!


## 8. 音频处理和转录

In [ ]:
def transcribe_sample(sample):
    """
    Revised transcription function using the dataset's built-in audio decoder.
    """
    try:
        # 1. Access the 'audio' key which datasets provides automatically
        # It contains {'array': np.array, 'sampling_rate': 16000}
        audio_data = sample['audio']
        
        if audio_data is None or len(audio_data['array']) == 0:
            return "", False
        
        # 2. Transcribe using the array directly
        # Whisper expects 16kHz, which 'audio' usually handles via the loader
        result = transcriber(audio_data["array"])
        transcription = result["text"].strip()
        
        # 3. Language Check
        is_eng = is_english(transcription)
        
        return transcription, is_eng
    
    except Exception as e:
        # Log the error so you know WHY it failed
        # tqdm.write(f"Error processing sample: {e}") 
        return "", False

In [ ]:
'''def decode_audio(wav_dict, target_sr=16000):
    """解码音频"""
    try:
        audio_bytes = wav_dict['bytes']
        audio_array, sample_rate = sf.read(
            io.BytesIO(audio_bytes),
            dtype='float32'
        )
        
        if sample_rate != target_sr:
            import librosa
            audio_array = librosa.resample(
                audio_array,
                orig_sr=sample_rate,
                target_sr=target_sr
            )
        
        return {'array': audio_array, 'sampling_rate': target_sr}
    except Exception as e:
        return None


def transcribe_sample(sample):
    """
    转录单个样本
    返回: (transcription, is_english_flag)
    """
    try:
        audio_data = decode_audio(sample['wav'])
        if audio_data is None:
            return "", False
        
        # 转录
        result = transcriber(audio_data)
        transcription = result["text"].strip()
        
        # 检查是否为英文
        is_eng = is_english(transcription)
        
        return transcription, is_eng
    
    except Exception as e:
        return "", False

print("✓ 音频处理函数已定义")'''

✓ 音频处理函数已定义


## 9. 主转录流程（带英文过滤）

In [9]:
# 检查断点
start_idx = 0
results = []

if RESUME:
    checkpoint = load_checkpoint()
    if checkpoint:
        results = checkpoint['results']
        start_idx = checkpoint['last_index'] + 1
        print(f"✓ 从检查点恢复: 已处理 {len(results)} 个样本")
        print(f"  从第 {start_idx + 1} 个样本继续\n")

# 开始转录
print("="*80)
print(f"开始转录: {start_idx + 1} 到 {len(dataset)} (共 {len(dataset) - start_idx} 个样本)")
print("="*80)
print()

# 统计
success_count = sum(1 for r in results if r['success'])
english_count = sum(1 for r in results if r.get('is_english', False))
non_english_filtered = 0

for i in tqdm(range(start_idx, len(dataset)), 
              initial=start_idx,
              total=len(dataset),
              desc="转录进度"):
    
    sample = dataset[i]
    
    # 获取参考文本
    reference_original = sample['text']
    
    # 转录（带英文检测）
    prediction_original, is_eng = transcribe_sample(sample)
    
    # 判断成功
    success = bool(prediction_original)
    
    # 如果启用过滤且不是英文，标记为失败
    if success and FILTER_NON_ENGLISH and not is_eng:
        non_english_filtered += 1
        success = False  # 标记为失败
        tqdm.write(f"⚠️  样本 {i}: 过滤非英文 - '{prediction_original[:50]}'")
    
    if success:
        success_count += 1
        if is_eng:
            english_count += 1
    
    # 保存结果
    result = {
        'index': i,
        'id': sample['ID'],
        'reference_text': reference_original,
        'transcription': prediction_original,
        'is_english': is_eng,
        'success': success
    }
    results.append(result)
    
    # 定期保存检查点
    if (i + 1) % SAVE_EVERY == 0:
        save_checkpoint(results, i)
        tqdm.write(f"✓ 检查点已保存 ({i + 1}/{len(dataset)})")

# 完成
print("\n" + "="*80)
print("转录完成!")
print("="*80)
print(f"总样本数: {len(results)}")
print(f"成功转录: {success_count} ({success_count/len(results)*100:.1f}%)")
print(f"英文样本: {english_count} ({english_count/len(results)*100:.1f}%)")
if FILTER_NON_ENGLISH:
    print(f"已过滤非英文: {non_english_filtered}")
print(f"失败数量: {len(results) - success_count}")

开始转录: 1 到 2000 (共 2000 个样本)



转录进度:   5%|▌         | 109/2000 [00:01<00:25, 73.77it/s]   

✓ 检查点已保存 (100/2000)


转录进度:  10%|█         | 208/2000 [00:02<00:23, 77.10it/s]    

✓ 检查点已保存 (200/2000)


...

✓ 检查点已保存 (1800/2000)


转录进度:  96%|█████████▌| 1913/2000 [00:26<00:01, 72.09it/s]    

✓ 检查点已保存 (1900/2000)


转录进度: 100%|██████████| 2000/2000 [00:28<00:00, 71.12it/s]    

✓ 检查点已保存 (2000/2000)

转录完成!
总样本数: 2000
成功转录: 0 (0.0%)
英文样本: 0 (0.0%)
失败数量: 2000


## 10. 计算整体 WER 和 CER（仅英文样本）

In [10]:
print("\n" + "="*80)
print("计算评估指标")
print("="*80)

# 只使用成功且为英文的样本
references_norm = []
predictions_norm = []

for result in results:
    if result['success'] and result.get('is_english', True):
        ref_norm = normalize_text(result['reference_text'])
        pred_norm = normalize_text(result['transcription'])
        
        references_norm.append(ref_norm)
        predictions_norm.append(pred_norm)

# 计算 WER 和 CER
if len(predictions_norm) > 0:
    wer_score = wer(references_norm, predictions_norm)
    cer_score = cer(references_norm, predictions_norm)
    
    print(f"\n基于 {len(predictions_norm)} 个英文样本:")
    print(f"  WER (词错误率): {wer_score:.4f} ({wer_score*100:.2f}%)")
    print(f"  CER (字符错误率): {cer_score:.4f} ({cer_score*100:.2f}%)")
    
    # 保存指标
    metrics = {
        'wer': wer_score,
        'cer': cer_score,
        'total_samples': len(results),
        'successful_samples': success_count,
        'english_samples': len(predictions_norm),
        'success_rate': success_count / len(results),
        'english_rate': len(predictions_norm) / len(results)
    }
else:
    print("\n⚠️  没有成功的英文样本")
    metrics = None


计算评估指标

⚠️  没有成功的英文样本


## 11. 保存为 JSON

In [11]:
# 构建输出数据
output_data = {
    'metadata': {
        'created_at': datetime.now().isoformat(),
        'dataset': 'speechbrain/LoquaciousSet',
        'split': DATASET_SPLIT,
        'model': MODEL_NAME,
        'total_samples': len(results),
        'successful_transcriptions': success_count,
        'english_only': FILTER_NON_ENGLISH,
        'force_english_recognition': FORCE_ENGLISH,
        'metrics': metrics
    },
    'results': results
}

# 保存 JSON
print(f"\n正在保存 JSON: {OUTPUT_JSON}...")
with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

file_size = Path(OUTPUT_JSON).stat().st_size / (1024 * 1024)
print(f"✓ JSON 已保存")
print(f"  文件: {Path(OUTPUT_JSON).absolute()}")
print(f"  大小: {file_size:.2f} MB")


正在保存 JSON: transcription_results_2000_english.json...
✓ JSON 已保存
  文件: d:\MyProject\transcription_results_2000_english.json
  大小: 0.65 MB


## 12. 保存为 CSV

In [12]:
print(f"\n正在保存 CSV: {OUTPUT_CSV}...")

with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
    fieldnames = ['index', 'id', 'reference_text', 'transcription', 'is_english', 'success']
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    
    writer.writeheader()
    writer.writerows(results)

file_size = Path(OUTPUT_CSV).stat().st_size / (1024 * 1024)
print(f"✓ CSV 已保存")
print(f"  文件: {Path(OUTPUT_CSV).absolute()}")
print(f"  大小: {file_size:.2f} MB")


正在保存 CSV: transcription_results_2000_english.csv...
✓ CSV 已保存
  文件: d:\MyProject\transcription_results_2000_english.csv
  大小: 0.38 MB


## 13. 结果预览

In [13]:
print("\n" + "="*80)
print("结果预览")
print("="*80)

print("\n【前3个成功的英文样本】")
english_samples = [r for r in results if r['success'] and r.get('is_english', True)][:3]

for i, sample in enumerate(english_samples, 1):
    print(f"\n样本 {i}:")
    print(f"  ID: {sample['id']}")
    print(f"  原文: {sample['reference_text'][:80]}...")
    print(f"  转录: {sample['transcription'][:80]}...")
    print(f"  语言: {'英文 ✓' if sample['is_english'] else '非英文 ✗'}")
    
    # 计算匹配度
    ref_norm = normalize_text(sample['reference_text'])
    pred_norm = normalize_text(sample['transcription'])
    
    if ref_norm == pred_norm:
        print(f"  匹配: ✓ 完全匹配")
    else:
        sample_wer = wer(ref_norm, pred_norm)
        print(f"  匹配: WER {sample_wer:.2%}")

# 显示被过滤的样本
if FILTER_NON_ENGLISH:
    print("\n" + "="*80)
    print("【被过滤的非英文样本示例】")
    print("="*80)
    
    filtered_samples = [r for r in results if not r['success'] and r['transcription'] and not r.get('is_english', True)][:3]
    
    for i, sample in enumerate(filtered_samples, 1):
        print(f"\n样本 {i}:")
        print(f"  转录结果: {sample['transcription'][:100]}")
        print(f"  被过滤原因: 非英文识别结果")


结果预览

【前3个成功的英文样本】


## 14. 最终总结

In [14]:
print("\n" + "="*80)
print("Phase 1 完成总结")
print("="*80)

print(f"\n✓ 转录任务完成")
print(f"  处理样本: {len(results)}")
print(f"  成功转录: {success_count} ({success_count/len(results)*100:.1f}%)")
print(f"  英文样本: {english_count} ({english_count/len(results)*100:.1f}%)")

if FILTER_NON_ENGLISH:
    print(f"\n✓ 质量控制")
    print(f"  已过滤非英文: {non_english_filtered}")
    print(f"  最终有效样本: {success_count}")

if metrics:
    print(f"\n✓ 评估指标 (规范化后, 仅英文样本)")
    print(f"  WER: {metrics['wer']*100:.2f}%")
    print(f"  CER: {metrics['cer']*100:.2f}%")
    print(f"  英文样本数: {metrics['english_samples']}")

print(f"\n✓ 输出文件")
print(f"  JSON: {OUTPUT_JSON}")
print(f"  CSV: {OUTPUT_CSV}")

print(f"\n✓ 优化效果")
print(f"  样本数调整: 3000 → 2000")
print(f"  英文限制: {'已启用 ✓' if FORCE_ENGLISH else '未启用'}")
print(f"  质量过滤: {'已启用 ✓' if FILTER_NON_ENGLISH else '未启用'}")

print(f"\n下一步: Phase 2 - 词频分析和关键词提取")
print(f"  运行 'Phase2_Analysis.ipynb'")

# 清理检查点
try:
    Path(CHECKPOINT_FILE).unlink()
    print(f"\n✓ 检查点文件已清理")
except:
    pass

print("\n" + "="*80)


Phase 1 完成总结

✓ 转录任务完成
  处理样本: 2000
  成功转录: 0 (0.0%)
  英文样本: 0 (0.0%)

✓ 输出文件
  JSON: transcription_results_2000_english.json
  CSV: transcription_results_2000_english.csv

✓ 优化效果
  样本数调整: 3000 → 2000
  英文限制: 未启用
  质量过滤: 未启用

下一步: Phase 2 - 词频分析和关键词提取
  运行 'Phase2_Analysis.ipynb'

✓ 检查点文件已清理

